### Simple RAG Archi without Tools

In [1]:
import os
import glob
from dotenv import load_dotenv
from pathlib import Path
import gradio as gr
from openai import OpenAI

In [2]:
load_dotenv()
openai = OpenAI()

#### Existing Documents

In [3]:
# employee files
filenames = glob.glob("knowledge-base/employees/*")
filenames

['knowledge-base/employees\\Alex Chen.md',
 'knowledge-base/employees\\Alex Harper.md',
 'knowledge-base/employees\\Alex Thomson.md',
 'knowledge-base/employees\\Amanda Foster.md',
 'knowledge-base/employees\\Avery Lancaster.md',
 'knowledge-base/employees\\Brandon Walker.md',
 'knowledge-base/employees\\Carlos Rodriguez.md',
 'knowledge-base/employees\\Daniel Park.md',
 'knowledge-base/employees\\David Kim.md',
 'knowledge-base/employees\\Emily Carter.md',
 'knowledge-base/employees\\Emily Tran.md',
 'knowledge-base/employees\\James Wilson.md',
 'knowledge-base/employees\\Jennifer Adams.md',
 'knowledge-base/employees\\Jessica Liu.md',
 'knowledge-base/employees\\Jordan Blake.md',
 'knowledge-base/employees\\Jordan K. Bishop.md',
 'knowledge-base/employees\\Kevin Zhang.md',
 'knowledge-base/employees\\Lisa Anderson.md',
 'knowledge-base/employees\\Marcus Johnson.md',
 'knowledge-base/employees\\Maxine Thompson.md',
 'knowledge-base/employees\\Maya Thompson.md',
 "knowledge-base/employ

#### Create a Knowledge Base

In [4]:
### update knowledge space with employee information

In [5]:
knowledge = {}
for filename in filenames:
    name = Path(filename).stem.split(" ")[-1]
    with open(filename, "r", encoding="utf-8") as f:
        knowledge[name.lower()] = f.read()

In [6]:
knowledge.keys()

dict_keys(['chen', 'harper', 'thomson', 'foster', 'lancaster', 'walker', 'rodriguez', 'park', 'kim', 'carter', 'tran', 'wilson', 'adams', 'liu', 'blake', 'bishop', 'zhang', 'anderson', 'johnson', 'thompson', "o'brien", 'rivera', 'patel', 'spencer', 'sharma', 'martinez', 'greene', 'trenton', 'williams', 'brooks'])

**Note:** We've created knowledge base with last name of employees as key to the knowledge base

In [7]:
### update knowledge space with product information

In [8]:
# product files
filenames = glob.glob("knowledge-base/products//*")

In [9]:
for filename in filenames:
    name = Path(filename).stem.split(" ")[-1]
    with open(filename, "r", encoding="utf-8") as f:
        knowledge[name.lower()] = f.read()

In [10]:
knowledge.keys()

dict_keys(['chen', 'harper', 'thomson', 'foster', 'lancaster', 'walker', 'rodriguez', 'park', 'kim', 'carter', 'tran', 'wilson', 'adams', 'liu', 'blake', 'bishop', 'zhang', 'anderson', 'johnson', 'thompson', "o'brien", 'rivera', 'patel', 'spencer', 'sharma', 'martinez', 'greene', 'trenton', 'williams', 'brooks', 'bizllm', 'carllm', 'claimllm', 'healthllm', 'homellm', 'lifellm', 'markellm', 'rellm'])

**Note:** We've updated knowledge base similar to how employees were created in the  knowledge base

#### Extract Context from Knowlegde base

In [11]:
# brute force approach to getting relevant context to user's questions from the knowledge base
# this is what vector databases do
def get_relevant_context(message):
    texts = ''.join(ch for ch in message if ch.isalpha() or ch.isspace())
    words = texts.lower().split()
    return [knowledge[word] for word in words if word in knowledge]

In [12]:
message = "Who is Avery Lancaster?"

In [13]:
get_relevant_context(message)

["# Avery Lancaster\n\n## Summary\n- **Date of Birth**: March 15, 1985\n- **Job Title**: Co-Founder & Chief Executive Officer (CEO)\n- **Location**: San Francisco, California\n- **Current Salary**: $225,000  \n\n## Insurellm Career Progression\n- **2015 - Present**: Co-Founder & CEO  \n  Avery Lancaster co-founded Insurellm in 2015 and has since guided the company to its current position as a leading Insurance Tech provider. Avery is known for her innovative leadership strategies and risk management expertise that have catapulted the company into the mainstream insurance market.  \n\n- **2013 - 2015**: Senior Product Manager at Innovate Insurance Solutions  \n  Before launching Insurellm, Avery was a leading Senior Product Manager at Innovate Insurance Solutions, where she developed groundbreaking insurance products aimed at the tech sector.  \n\n- **2010 - 2013**: Business Analyst at Edge Analytics  \n  Prior to joining Innovate, Avery worked as a Business Analyst, focusing on market 

In [14]:
get_relevant_context("Who is Lancaster and what is Carllm?")

["# Avery Lancaster\n\n## Summary\n- **Date of Birth**: March 15, 1985\n- **Job Title**: Co-Founder & Chief Executive Officer (CEO)\n- **Location**: San Francisco, California\n- **Current Salary**: $225,000  \n\n## Insurellm Career Progression\n- **2015 - Present**: Co-Founder & CEO  \n  Avery Lancaster co-founded Insurellm in 2015 and has since guided the company to its current position as a leading Insurance Tech provider. Avery is known for her innovative leadership strategies and risk management expertise that have catapulted the company into the mainstream insurance market.  \n\n- **2013 - 2015**: Senior Product Manager at Innovate Insurance Solutions  \n  Before launching Insurellm, Avery was a leading Senior Product Manager at Innovate Insurance Solutions, where she developed groundbreaking insurance products aimed at the tech sector.  \n\n- **2010 - 2013**: Business Analyst at Edge Analytics  \n  Prior to joining Innovate, Avery worked as a Business Analyst, focusing on market 

**Recall:** We've created knowledge base with last name of employees. Here is where "tricks" like this fails and why we need a vector database that can create vector for anything (people, products, policies etc)


if we try to search for relevant context using Avery, we would get nothing. Although we could use first name and last name but this is not effective.

In [15]:
get_relevant_context("Who is Avery?")

[]

In [16]:
def get_additional_context(message):
    relevent_context = get_relevant_context(message)
    if not relevent_context:
        return "No relevant context to answer this user"
    else:
        result = "The following context may be relevant to answering the user's question:\n\n"
        result +=  "\n\n".join(relevent_context)
    return result

In [17]:
get_additional_context("Who is Avery?")

'No relevant context to answer this user'

In [18]:
print(get_additional_context("Who is Avery Lancaster?"))

The following context may be relevant to answering the user's question:

# Avery Lancaster

## Summary
- **Date of Birth**: March 15, 1985
- **Job Title**: Co-Founder & Chief Executive Officer (CEO)
- **Location**: San Francisco, California
- **Current Salary**: $225,000  

## Insurellm Career Progression
- **2015 - Present**: Co-Founder & CEO  
  Avery Lancaster co-founded Insurellm in 2015 and has since guided the company to its current position as a leading Insurance Tech provider. Avery is known for her innovative leadership strategies and risk management expertise that have catapulted the company into the mainstream insurance market.  

- **2013 - 2015**: Senior Product Manager at Innovate Insurance Solutions  
  Before launching Insurellm, Avery was a leading Senior Product Manager at Innovate Insurance Solutions, where she developed groundbreaking insurance products aimed at the tech sector.  

- **2010 - 2013**: Business Analyst at Edge Analytics  
  Prior to joining Innovate, 

In [19]:
### Another challenge with brute force knowledge base approach
print(get_additional_context("Who is Alex Lancaster?"))

The following context may be relevant to answering the user's question:

# Avery Lancaster

## Summary
- **Date of Birth**: March 15, 1985
- **Job Title**: Co-Founder & Chief Executive Officer (CEO)
- **Location**: San Francisco, California
- **Current Salary**: $225,000  

## Insurellm Career Progression
- **2015 - Present**: Co-Founder & CEO  
  Avery Lancaster co-founded Insurellm in 2015 and has since guided the company to its current position as a leading Insurance Tech provider. Avery is known for her innovative leadership strategies and risk management expertise that have catapulted the company into the mainstream insurance market.  

- **2013 - 2015**: Senior Product Manager at Innovate Insurance Solutions  
  Before launching Insurellm, Avery was a leading Senior Product Manager at Innovate Insurance Solutions, where she developed groundbreaking insurance products aimed at the tech sector.  

- **2010 - 2013**: Business Analyst at Edge Analytics  
  Prior to joining Innovate, 

#### System Prompt to Accompany Relevent Contexts based on user input (message)

In [20]:
SYSTEM_PREFIX =  """
You represent insurellm, the Insurance Tech Company.
You are an expert in answering questions about Insurellm; its employees and its products.
You are provided with additional context that might be relevant to the user's questions.
Give brief, accurate answers. If you don't know the answer, say so.

Relevant Context:
"""

In [21]:
MODEL = "gpt-4.1-nano"

In [22]:
def chat(message, history):
    system_message= SYSTEM_PREFIX + get_additional_context(message)
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages)
    return response.choices[0].message.content

### Using Gradio Chat Interface

In [23]:
view = gr.ChatInterface(chat).launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
